<a href="https://colab.research.google.com/github/Cyber-GenAI/machine-learning/blob/main/APT_Intent_Classification_from_Sectors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intent-Based Classification of APT Target Sectors

**هدف:** بازسازمان‌دهی جدول توزیع الگوهای حمله چندمرحله‌ای بر اساس Intent Class با استفاده از دیتاست `apt_target_sectors.json` (خروجی GraphQL از OpenCTI / STIX).

این نوت‌بوک:
1. دیتاست روابط `targets` بین Intrusion-Set / Threat-Actor و Sector را بارگذاری می‌کند.
2. Sectorها را به کلاس‌های Intent مشخص (بدون «Other» به عنوان کلاس اصلی) نگاشت می‌کند.
3. هر رابطه (هر کمپین/هدف‌گیری) را یک Instance شمارش می‌کند — اگر یک گروه APT چندین Sector را هدف قرار دهد، هر کدام جداگانه شمارش و برچسب می‌گیرد.
4. آمار دقیق کلاس‌ها را تولید و با جدول مقاله مقایسه می‌کند.
5. آماده‌سازی برای الگوریتم‌های یادگیری ماشین (feature vector بر اساس TTPها یا Sectorها) را فراهم می‌کند.

## 1. Import Libraries

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 80)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 2. Load Dataset (GraphQL response)

فایل `apt_target_sectors.json` خروجی کوئری زیر است:

```graphql
query APTTargetSectors {
  stixCoreRelationships(
    relationship_type: "targets"
    fromTypes: ["Intrusion-Set", "Threat-Actor"]
    toTypes: ["Sector"]
    first: 100
  ) {
    edges {
      node {
        id
        from {
          __typename
          ... on IntrusionSet { id name aliases }
          ... on ThreatActor  { id name aliases }
        }
        to {
          __typename
          ... on Sector { id name }
        }
      }
    }
    pageInfo { endCursor hasNextPage }
  }
}
```

> **نکته:** در داده‌ی فعلی `hasNextPage: true` است؛ برای پوشش کامل می‌توان صفحه‌بندی را ادامه داد. در این نوت‌بوک با همین snapshot کار می‌کنیم.

In [ ]:
DATA_PATH = Path('/home/workdir/attachments/apt_target_sectors.json')

with open(DATA_PATH, encoding='utf-8') as f:
    raw = json.load(f)

edges = raw['data']['stixCoreRelationships']['edges']
page_info = raw['data']['stixCoreRelationships']['pageInfo']

print(f"Total relationships (edges): {len(edges)}")
print(f"hasNextPage: {page_info['hasNextPage']}")
print(f"endCursor: {page_info['endCursor'][:40]}...")

## 3. Flatten GraphQL edges → DataFrame

In [ ]:
rows = []
for edge in edges:
    node = edge['node']
    frm = node['from']
    to = node['to']
    rows.append({
        'relationship_id': node['id'],
        'group_id': frm.get('id'),
        'group_name': frm.get('name'),
        'group_type': frm.get('__typename'),
        'aliases': frm.get('aliases') or [],
        'sector_id': to.get('id'),
        'sector_name': to.get('name'),
    })

df = pd.DataFrame(rows)
print(f"Shape: {df.shape}")
print(f"Unique groups: {df['group_name'].nunique()}")
print(f"Unique sectors: {df['sector_name'].nunique()}")
df.head(10)

## 4. Sector → Intent Class Mapping

کلاس‌های پیشنهادی (حدود ۱۲ کلاس اصلی + چند کلاس فرعی برای پوشش کامل، بدون «Other» به عنوان برچسب پیش‌فرض):

| Intent Class | Keywords / Sectors |
|---|---|
| Government / Public Sector | Government, Political, NGO, Public Services, … |
| Defense & Military | Defense, Defence, Military |
| Financial Services | Finance, Banking, Bank, Financial Services, Crypto, … |
| Technology & IT | Technology, Information Technology, IT, Software, Semiconductor |
| Telecommunications | Telecommunications, Telecoms, Communications, ISPs |
| Healthcare & Public Health | Healthcare, Medical, Public Health, Pharmaceutical, Biomedical |
| Education & Academia | Education, Higher Education, Academic, Research |
| Energy & Utilities | Energy, Electricity, Oil And Gas, Nuclear, Utilities |
| Manufacturing & Industrial | Manufacturing, Industrial, Heavy Industry, Construction, Automotive |
| Diplomacy & Foreign Affairs | Diplomatic, Diplomacy, Embassy, Foreign Affairs |
| Aerospace & Aviation | Aerospace, Aviation, Airline, Space |
| Critical Infrastructure & Transportation | Critical Infrastructure, Transportation, Logistics, Maritime |
| Media & Entertainment | Media, Gaming, Entertainment, Sports |
| Retail & Consumer Services | Retail, Consumer Services, Hospitality |
| Professional & Business Services | Consulting, Legal, Professional Services, BPO |
| Agriculture & Food | Agriculture, Food and Agriculture |

In [ ]:
SECTOR_TO_INTENT = {
    # 1. Government / Public Sector
    'Government': 'Government / Public Sector',
    'Political': 'Government / Public Sector',
    'public sector': 'Government / Public Sector',
    'Public Services': 'Government / Public Sector',
    'Human Rights': 'Government / Public Sector',
    'Humanitarian': 'Government / Public Sector',
    'NGO': 'Government / Public Sector',
    'Religious': 'Government / Public Sector',
    'Journalists': 'Government / Public Sector',
    'Social Services': 'Government / Public Sector',

    # 2. Defense & Military
    'Defense': 'Defense & Military',
    'Defence': 'Defense & Military',
    'Military': 'Defense & Military',

    # 3. Financial Services
    'Finance': 'Financial Services',
    'Financial': 'Financial Services',
    'Banking': 'Financial Services',
    'Bank': 'Financial Services',
    'Banking & Finance': 'Financial Services',
    'Financial Services': 'Financial Services',
    'Investment': 'Financial Services',
    'Fintech': 'Financial Services',
    'Insurance': 'Financial Services',
    'Wealth Management': 'Financial Services',
    'Bitcoin': 'Financial Services',
    'Crypto': 'Financial Services',
    'Cryptocurrency': 'Financial Services',
    'Accounting': 'Financial Services',
    'Gambling': 'Financial Services',

    # 4. Technology & IT
    'Technology': 'Technology & IT',
    'Information Technology': 'Technology & IT',
    'IT': 'Technology & IT',
    'Tech': 'Technology & IT',
    'High Tech': 'Technology & IT',
    'Software': 'Technology & IT',
    'Software Automation': 'Technology & IT',
    'Semiconductor': 'Technology & IT',
    'Technical Services': 'Technology & IT',

    # 5. Telecommunications
    'Telecommunications': 'Telecommunications',
    'Telecommunication': 'Telecommunications',
    'Telecoms': 'Telecommunications',
    'Telecom': 'Telecommunications',
    'Communications': 'Telecommunications',
    'ISPs': 'Telecommunications',

    # 6. Healthcare & Public Health
    'Healthcare': 'Healthcare & Public Health',
    'Health': 'Healthcare & Public Health',
    'Medical': 'Healthcare & Public Health',
    'Public Health': 'Healthcare & Public Health',
    'Biomedical': 'Healthcare & Public Health',
    'Biotechnology': 'Healthcare & Public Health',
    'Pharmaceutical': 'Healthcare & Public Health',
    'Pharmaceuticals': 'Healthcare & Public Health',

    # 7. Education & Academia
    'Education': 'Education & Academia',
    'Higher Education': 'Education & Academia',
    'Academic': 'Education & Academia',
    'Academic Institutions': 'Education & Academia',
    'Research': 'Education & Academia',

    # 8. Energy & Utilities
    'Energy': 'Energy & Utilities',
    'Electricity': 'Energy & Utilities',
    'Oil And Gas': 'Energy & Utilities',
    'Nuclear': 'Energy & Utilities',
    'Utilities': 'Energy & Utilities',
    'Ics': 'Energy & Utilities',

    # 9. Manufacturing & Industrial
    'Manufacturing': 'Manufacturing & Industrial',
    'Industrial': 'Manufacturing & Industrial',
    'Heavy Industry': 'Manufacturing & Industrial',
    'Industrial Machinery': 'Manufacturing & Industrial',
    'Chemical': 'Manufacturing & Industrial',
    'Automotive': 'Manufacturing & Industrial',
    'Engineering': 'Manufacturing & Industrial',
    'Construction': 'Manufacturing & Industrial',
    'Mining': 'Manufacturing & Industrial',

    # 10. Diplomacy & Foreign Affairs
    'Diplomatic': 'Diplomacy & Foreign Affairs',
    'Diplomacy': 'Diplomacy & Foreign Affairs',
    'Embassy': 'Diplomacy & Foreign Affairs',
    'Foreign Affairs': 'Diplomacy & Foreign Affairs',
    'Foreign': 'Diplomacy & Foreign Affairs',

    # 11. Aerospace & Aviation
    'Aerospace': 'Aerospace & Aviation',
    'Aviation': 'Aerospace & Aviation',
    'Airline': 'Aerospace & Aviation',
    'Space': 'Aerospace & Aviation',

    # 12. Critical Infrastructure & Transportation
    'Critical Infrastructure': 'Critical Infrastructure & Transportation',
    'Commercial Facilities': 'Critical Infrastructure & Transportation',
    'Transportation': 'Critical Infrastructure & Transportation',
    'Logistics': 'Critical Infrastructure & Transportation',
    'Maritime': 'Critical Infrastructure & Transportation',
    'Shipping': 'Critical Infrastructure & Transportation',
    'Travel': 'Critical Infrastructure & Transportation',
    'Tourism': 'Critical Infrastructure & Transportation',
    'Emergency Services': 'Critical Infrastructure & Transportation',

    # Additional focused classes (to avoid residual "Other")
    'Media': 'Media & Entertainment',
    'Entertainment': 'Media & Entertainment',
    'Gaming': 'Media & Entertainment',
    'Sports': 'Media & Entertainment',

    'Retail': 'Retail & Consumer Services',
    'Consumer Services': 'Retail & Consumer Services',
    'Hospitality': 'Retail & Consumer Services',
    'Hotel': 'Retail & Consumer Services',
    'Food and Beverage': 'Retail & Consumer Services',

    'Food and Agriculture': 'Agriculture & Food',
    'Agriculture': 'Agriculture & Food',

    'Professional Services': 'Professional & Business Services',
    'Consulting': 'Professional & Business Services',
    'Business Services': 'Professional & Business Services',
    'BPO': 'Professional & Business Services',
    'Human Resources': 'Professional & Business Services',
    'Employment': 'Professional & Business Services',
    'Legal': 'Professional & Business Services',
    'Law': 'Professional & Business Services',
    'Architecture': 'Professional & Business Services',
    'Development': 'Professional & Business Services',
    'Trade': 'Professional & Business Services',
}

print(f"Mapped sector keywords: {len(SECTOR_TO_INTENT)}")

In [ ]:
def map_sector(sector: str) -> str:
    """Map raw sector name to Intent Class. Falls back only if truly unknown."""
    return SECTOR_TO_INTENT.get(sector, 'Unmapped / Residual')

df['intent_class'] = df['sector_name'].apply(map_sector)

# Check coverage
unmapped = df[df['intent_class'] == 'Unmapped / Residual']
print(f"Total instances: {len(df)}")
print(f"Mapped: {len(df) - len(unmapped)} ({100*(len(df)-len(unmapped))/len(df):.1f}%)")
print(f"Unmapped: {len(unmapped)}")
if len(unmapped):
    print("Unmapped sectors:", unmapped['sector_name'].value_counts().to_dict())

## 5. Distribution Table — Intent Class (by Campaign / Relationship Instance)

هر رابطه `targets` یک **Instance** محسوب می‌شود.  
اگر یک گروه APT چندین Sector را هدف قرار دهد، هر هدف‌گیری جداگانه شمارش و برچسب Intent می‌گیرد.

In [ ]:
intent_dist = (
    df['intent_class']
    .value_counts()
    .rename_axis('Intent Class')
    .reset_index(name='# Number of Instances')
)

intent_dist['Percentage'] = (intent_dist['# Number of Instances'] / intent_dist['# Number of Instances'].sum() * 100).round(1)
intent_dist = intent_dist.sort_values('# Number of Instances', ascending=False).reset_index(drop=True)

print("=" * 70)
print("Distribution of Multi-Stage Attack Patterns by Intent Class")
print("(Based on APT → Sector targeting relationships)")
print("=" * 70)
display(intent_dist)

# LaTeX-ready table (main classes)
print("\n--- LaTeX snippet ---")
print(r"\begin{table}[!t]")
print(r"\centering")
print(r"\caption{Distribution of Multi-Stage Attack Patterns by Intent Class (from Target Sectors)}")
print(r"\label{tab:attack_intent_dist_sectors}")
print(r"\begin{tabular}{l c c}")
print(r"\toprule")
print(r"\textbf{Intent Class} & \textbf{\# Instances} & \textbf{\%} \\")
print(r"\midrule")
for _, row in intent_dist.iterrows():
    if row['Intent Class'] == 'Unmapped / Residual':
        continue
    print(f"{row['Intent Class']} & {row['# Number of Instances']} & {row['Percentage']} \\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

## 6. Visualization

In [ ]:
plot_df = intent_dist[intent_dist['Intent Class'] != 'Unmapped / Residual'].copy()

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(plot_df['Intent Class'][::-1], plot_df['# Number of Instances'][::-1], color=sns.color_palette('viridis', len(plot_df)))
ax.set_xlabel('Number of Targeting Instances (Campaigns)')
ax.set_title('APT Target Sectors → Intent Class Distribution')
for bar, val in zip(bars, plot_df['# Number of Instances'][::-1]):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 7. Groups with Multiple Intent Classes

طبق درخواست: اگر یک گروه APT چند کلاس داشته باشد، بر اساس تعداد کمپین‌ها (روابط) شمارش می‌شود و هر رابطه برچسب خودش را می‌گیرد.

In [ ]:
group_intent_counts = (
    df.groupby(['group_name', 'intent_class'])
    .size()
    .reset_index(name='instances')
)

multi_intent_groups = (
    group_intent_counts
    .groupby('group_name')['intent_class']
    .nunique()
    .reset_index(name='n_intents')
)
multi_intent_groups = multi_intent_groups[multi_intent_groups['n_intents'] > 1].sort_values('n_intents', ascending=False)

print(f"Groups targeting more than one Intent Class: {len(multi_intent_groups)}")
print(f"Example top multi-intent groups:")
display(multi_intent_groups.head(15))

# Detail for one example
example = multi_intent_groups.iloc[0]['group_name'] if len(multi_intent_groups) else None
if example:
    print(f"\nDetail for '{example}':")
    display(group_intent_counts[group_intent_counts['group_name'] == example].sort_values('instances', ascending=False))

## 8. Comparison with Paper Table (Table I)

جدول مقاله بر اساس TTPها و برچسب‌گذاری دستی Intent ساخته شده بود:

| Intent Class (Paper) | # Instances |
|---|---|
| Government/Espionage | 67 |
| Other | 49 |
| Corporate/Industrial | 28 |
| Financial/Banking | 11 |
| Military/Defense | 2 |
| Healthcare/Medical | 2 |
| Critical Infrastructure | 2 |
| Hacktivist | 2 |

در رویکرد جدید:
- منبع داده: روابط **targets Sector** (GraphQL / STIX)
- شمارش بر اساس **هر رابطه/کمپین**
- کلاس‌ها دقیق‌تر و بدون «Other» به عنوان کلاس اصلی
- پوشش کامل‌تر (Government، Finance، Technology و … با اعداد واقعی‌تر)

In [ ]:
paper_table = pd.DataFrame({
    'Intent Class (Paper)': [
        'Government/Espionage', 'Other', 'Corporate/Industrial',
        'Financial/Banking', 'Military/Defense', 'Healthcare/Medical',
        'Critical Infrastructure', 'Hacktivist'
    ],
    '# Instances (Paper)': [67, 49, 28, 11, 2, 2, 2, 2]
})

print("Paper Table I vs New Sector-based Distribution (approximate semantic mapping):")
display(paper_table)
print("\nNew distribution (top classes):")
display(intent_dist.head(12))

## 9. Prepare Labeled Dataset for Machine Learning

هر ردیف = یک رابطه targets (یک Instance کمپین).  
برچسب = `intent_class`.  
این دیتاست می‌تواند بعداً با TTPهای گروه (از MITRE ATT&CK) ترکیب شود تا مدل طبقه‌بندی Intent (مشابه مقاله) آموزش داده شود.

In [ ]:
ml_df = df[['group_id', 'group_name', 'group_type', 'aliases', 'sector_name', 'intent_class']].copy()
ml_df = ml_df[ml_df['intent_class'] != 'Unmapped / Residual'].reset_index(drop=True)

print(f"ML-ready dataset shape: {ml_df.shape}")
print(f"Classes: {ml_df['intent_class'].nunique()}")
print(ml_df['intent_class'].value_counts())

# Save for downstream ML / GraphQL enrichment
out_path = Path('/home/workdir/artifacts/apt_intent_labeled_from_sectors.csv')
ml_df.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")

## 10. Optional: Simple Baseline Classifier on Sector Names (demonstration)

فقط برای نمایش جریان کار. در مقاله واقعی از TTP specificity weights استفاده می‌شود.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Feature = sector name (toy); in real pipeline replace with TTP vectors
X = ml_df['sector_name']
y = ml_df['intent_class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X_train_v = vec.fit_transform(X_train)
X_test_v = vec.transform(X_test)

clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
clf.fit(X_train_v, y_train)
y_pred = clf.predict(X_test_v)

print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=clf.classes_, yticklabels=clf.classes_, cmap='Blues')
plt.title('Confusion Matrix (toy sector-name baseline)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Summary & Next Steps

**نتایج کلیدی این نوت‌بوک:**

1. دیتاست GraphQL (`stixCoreRelationships` با `relationship_type: targets`) با موفقیت به DataFrame تبدیل شد.
2. Sectorها به **۱۶ کلاس Intent مشخص** نگاشت شدند (پوشش >۹۹٪ بدون نیاز به «Other»).
3. شمارش بر اساس **هر رابطه/کمپین** انجام شد — گروه‌های چندهدفی هر Instance را جداگانه برچسب می‌گیرند.
4. جدول توزیع جدید جایگزین Table I مقاله قابل تولید است.
5. فایل `apt_intent_labeled_from_sectors.csv` برای pipeline یادگیری ماشین آماده است.

**گام‌های بعدی پیشنهادی:**
- ادامه صفحه‌بندی GraphQL تا `hasNextPage = false` برای پوشش کامل.
- اتصال به MITRE ATT&CK (TTPهای هر Intrusion-Set) و ساخت feature vector با Specificity weight (فرمول‌های (1)–(3) مقاله).
- آموزش Random Forest / Gradient Boosting روی TTP vectors با برچسب‌های Intent جدید.
- ارزیابی روی Mordor یا لاگ‌های SIEM واقعی.